In [1]:
import pandas as pd 
import glob
import os

In [2]:
MAPEAMENTO_REGIOES = {
    'Zona Sul': ['Ipanema','Leblon','Botafogo','Copacabana','Gávea','Lagoa','Leme','Flamengo','Cosme Velho','Humaitá','São Conrado','Jardim Botânico','Rocinha','Vidigal','Urca','Laranjeiras','Catete','Glória','Santa Teresa','Joá'],
    'Centro': ['Centro','Cidade Nova','Santo Cristo','Gamboa','Saúde','Rio Comprido','Imperial de São Cristóvão','Praça da Bandeira','Catumbi','Estácio','Lapa'],
    'Zona Oeste': ['Campo Grande','Santa Cruz','Bangu','Realengo','Padre Miguel','Sepetiba','Guaratiba','Paciência','Santíssimo','Inhoaíba','Cosmos','Pedra de Guaratiba','Ilha de Guaratiba','Jabour','Jardim América','Magalhães Bastos','Senador Camará','Senador Vasconcelos','Vila Kennedy','Deodoro','Campo dos Afonsos','Vila Militar','Anchieta','Recreio dos Bandeirantes','Barra da Tijuca','Barra de Guaratiba','Barra Olímpica','Vargem Grande','Vargem Pequena','Camorim','Curicica','Gardênia Azul','Itanhangá','Jacarepaguá','Pechincha','Praça Seca','Taquara','Tanque','Anil','Cidade de Deus','Jardim Sulacap','Vila Valqueire','Freguesia (Jacarepaguá)'],
    'Zona Norte / Subúrbio': ['Madureira','Cascadura','Irajá','Pavuna','Marechal Hermes','Méier','Engenho de Dentro','Engenho da Rainha','Engenho Novo','Bonsucesso','Ramos','Olaria','Penha','Cordovil','Guadalupe','Rocha Miranda','Honório Gurgel','Coelho Neto','Bento Ribeiro','Del Castilho','Inhaúma','Acari','Água Santa','Grajaú','Vila Isabel','Andaraí','Tijuca','Alto da Boa Vista','Cachambi','Campinho','Encantado','Lins de Vasconcelos','Pilares','Piedade','Quintino Bocaiúva','Riachuelo','Rocha','Sampaio','Todos os Santos','Tomás Coelho','Turiaçu','Osvaldo Cruz','Barros Filho','Colégio','Costa Barros','Engenheiro Leal','Higienópolis','Parque Colúmbia','Parada de Lucas','Ricardo de Albuquerque','Vaz Lobo','Vicente de Carvalho','Vila da Penha','Vista Alegre','Vigário Geral','Complexo do Alemão','Jacarezinho','Manguinhos','Benfica','Caju','Mangueira','Maracanã','São Francisco Xavier','Jacaré','Brás de Pina','Cavalcanti','Penha Circular','Vasco da Gama','Maria da Graça','Bancários','Cacuia','Cocotá','Galeão','Jardim Carioca','Jardim Guanabara','Moneró','Pitangueiras','Portuguesa','Praia da Bandeira','Ribeira','Tauá','Zumbi','Maré','Abolição', 'Cidade Universitária']
}

# transforma o {regiao: [bairros]} em {bairro: regiao}
BAIRRO_TO_REGIAO = {bairro: regiao for regiao, lista in MAPEAMENTO_REGIOES.items() for bairro in lista}

Abaixo estão definidas as funções que tratam cada .csv de forma individual. No dicionário mapa_tratamento encontra-se o caminho do arquivo a qual cada função se refere.

In [3]:
def tratar_ids(df):
    return df[['codbairro', 'bairro', 'ids']]

# def tratar_lim_bairros(df):
#    ...

def tratar_lotacao_onibus(df):
    df['servico'] = df['servico'].astype(str)                                   # servico (linha) está como número mas é pra ser interpretado como string
    df = df[(df['servico'] !='795') & (~df['servico'].str.contains('LECD'))]
    df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')                  # coloca a coluna data com valor de data no formato certo
    df = df[df['class_servico'] != 'Rodoviário']                                # remove do lotação onibus rodoviários
    return df

def tratar_matriz_lb(df):
    df['linha'] = df['linha'].astype(str)

    # cria a coluna regiao mapeando pelo dicionário, e preenche os nan com "Não identificado"
    df['regiao'] = df['bairro'].map(BAIRRO_TO_REGIAO).fillna('Não identificado')

    return df

def tratar_sumario(df):
    df['data'] = pd.to_datetime(df['data'], format='%Y-%m-%d')
    df = df[df['km_planejada'] > 0]                                             # linha não era pra operar no dia
    df = df[['servico', 'data', 'perc_km_planejada']]                           # só precisa dessas 3 colunas
    return df

mapa_tratamento = {
    'ids_por_bairro_rj.csv': tratar_ids,
    # 'limite_de_bairros.csv': tratar_lim_bairros,
    'lotacao_onibus.csv': tratar_lotacao_onibus,
    'matriz_linha_bairro.csv': tratar_matriz_lb,
    'sumario_servico_dia_tipo.csv': tratar_sumario,
}


Lendo os arquivos e realizando o tratamento adequado, armazenando os DataFrames no dicionário datasets. 
As chaves de acesso são os nomes do arquivo sem a extensão.

In [4]:
datasets = {}

# daqui em diante ele vai tratar tudo e colocar no dicionário datasets criado ali em cima
arquivos = glob.glob('../dados/**/*.csv', recursive=True)

for caminho in arquivos:
    nome_arquivo = os.path.basename(caminho)

    nome_df = os.path.splitext(nome_arquivo)[0]
    
    df = pd.read_csv(caminho)
    
    if nome_arquivo in mapa_tratamento:
        df = mapa_tratamento[nome_arquivo](df)
    
        datasets[nome_df] = df

Salvando os arquivos tratados no formato csv. Eles serão lidos no próximo notebook, o 02_eda.ipynb.

In [5]:
def salvar_datasets(dicionario_dfs, mapa_chaves, pasta_destino='../dados_tratados'):

    if not os.path.exists(pasta_destino):
        os.makedirs(pasta_destino)
        print(f"Pasta '{pasta_destino}' criada")

    chaves_validas = [os.path.splitext(arq)[0] for arq in mapa_chaves.keys()]

    for nome, df in dicionario_dfs.items():
        if nome in chaves_validas:
            caminho_arquivo = os.path.join(pasta_destino, f"{nome}_tratado.csv")
        
            try:
                df.to_csv(caminho_arquivo, index=False, encoding='utf-8')
                print(f"{nome}_tratado.csv salvo.")
            except Exception as e:
                print(f"Erro ao salvar o arquivo {nome}: {e}")

salvar_datasets(datasets, mapa_tratamento)

matriz_linha_bairro_tratado.csv salvo.
lotacao_onibus_tratado.csv salvo.
ids_por_bairro_rj_tratado.csv salvo.
sumario_servico_dia_tipo_tratado.csv salvo.
